In [22]:
import os
import sys
import json
import re
import subprocess
from pathlib import Path
import asyncio
import nest_asyncio
from openai import AsyncOpenAI

nest_asyncio.apply()

# ---------------------------------------------------------------------------
# 1. Environment and Client Setup
# ---------------------------------------------------------------------------
WORKSPACE_DIR = Path("code").resolve()
WORKSPACE_DIR.mkdir(parents=True, exist_ok=True)

OLLAMA_CLIENT = AsyncOpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",
)

# ---------------------------------------------------------------------------
# 2. Workspace Tools
# ---------------------------------------------------------------------------

def read_workspace_file(file_path: str) -> str:
    """Reads text content from a file in the workspace."""
    target = (WORKSPACE_DIR / file_path).resolve()
    if not str(target).startswith(str(WORKSPACE_DIR)):
        return "Error: Access denied. Path outside allowed workspace."
    if not target.exists():
        return f"Error: File '{file_path}' does not exist."
    try:
        return target.read_text(encoding="utf-8")
    except Exception as e:
        return f"Error reading file '{file_path}': {str(e)}"

def write_workspace_file(file_path: str, content: str) -> str:
    """Overwrites or creates a file with updated code inside the workspace."""
    target = (WORKSPACE_DIR / file_path).resolve()
    if not str(target).startswith(str(WORKSPACE_DIR)):
        return "Error: Access denied. Path outside allowed workspace."
    try:
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_text(content, encoding="utf-8")
        return f"Successfully updated and saved '{file_path}'."
    except Exception as e:
        return f"Error writing file '{file_path}': {str(e)}"

def run_python_script(file_path: str, timeout_seconds: int = 15) -> str:
    """Executes a Python script in a subprocess and captures standard output and errors."""
    target = (WORKSPACE_DIR / file_path).resolve()
    if not str(target).startswith(str(WORKSPACE_DIR)):
        return "Error: Access denied. Path outside allowed workspace."
    if not target.exists():
        return f"Error: Script '{file_path}' not found."

    try:
        result = subprocess.run(
            [sys.executable, str(target)],
            capture_output=True,
            text=True,
            timeout=timeout_seconds,
            cwd=str(WORKSPACE_DIR),
        )
        
        output = []
        if result.stdout.strip():
            output.append(f"[STDOUT]:\n{result.stdout.strip()}")
        if result.stderr.strip():
            output.append(f"[STDERR / TRACEBACK]:\n{result.stderr.strip()}")

        if result.returncode == 0 and not result.stderr:
            return "[EXECUTION SUCCESSFUL]\n" + ("\n".join(output) if output else "Script executed cleanly.")
        else:
            return f"[EXECUTION FAILED - Exit Code {result.returncode}]\n" + "\n".join(output)

    except subprocess.TimeoutExpired:
        return f"Error: Execution timed out after {timeout_seconds} seconds."
    except Exception as e:
        return f"Execution error: {str(e)}"

TOOL_REGISTRY = {
    "read_workspace_file": read_workspace_file,
    "write_workspace_file": write_workspace_file,
    "run_python_script": run_python_script,
}

TOOLS_SCHEMA = [
    {
        "type": "function",
        "function": {
            "name": "read_workspace_file",
            "description": "Reads full content from a target file in the workspace.",
            "parameters": {
                "type": "object",
                "properties": {"file_path": {"type": "string"}},
                "required": ["file_path"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "write_workspace_file",
            "description": "Mandatory tool to save modified or fixed python code to disk.",
            "parameters": {
                "type": "object",
                "properties": {
                    "file_path": {"type": "string"},
                    "content": {"type": "string"},
                },
                "required": ["file_path", "content"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "run_python_script",
            "description": "Executes a Python script to verify runtime correctness and capture tracebacks.",
            "parameters": {
                "type": "object",
                "properties": {"file_path": {"type": "string"}},
                "required": ["file_path"],
            },
        },
    },
]

# ---------------------------------------------------------------------------
# 3. Fallback & Extraction Parsing
# ---------------------------------------------------------------------------

def extract_fallback_tool_call(content: str):
    """Extracts raw JSON function calls embedded in conversational text."""
    if not content:
        return None, None
    try:
        match = re.search(r"\{.*\"name\"\s*:\s*\"([^\"]+)\".*\}", content, re.DOTALL)
        if match:
            data = json.loads(match.group(0))
            if "name" in data and "arguments" in data:
                return data["name"], data["arguments"]
    except Exception:
        pass
    return None, None

def extract_python_code_block(content: str) -> str | None:
    """Extracts raw python code blocks when the 7B model dumps code in markdown."""
    if not content:
        return None
    match = re.search(r"```python\s*(.*?)\s*```", content, re.DOTALL)
    if match:
        return match.group(1).strip()
    return None

# ---------------------------------------------------------------------------
# 4. Agentic ReAct Engine with Heuristic Code Capture
# ---------------------------------------------------------------------------

async def inspect_and_fix_code(target_file: str, max_turns: int = 10):
    system_instruction = (
        "You are an autonomous Senior Software Engineer.\n"
        "Execution Protocol:\n"
        "1. First, call 'read_workspace_file' to inspect the code.\n"
        "2. Call 'run_python_script' to reproduce any runtime errors.\n"
        "3. If errors or logic bugs exist, YOU MUST INVOKE 'write_workspace_file' to apply fixes to disk. "
        "DO NOT just output code blocks in text format.\n"
        "4. After calling 'write_workspace_file', call 'run_python_script' again to verify execution.\n"
        "5. Conclude only after code is written to disk and verified."
        "6. Write comment whererever you make changes to the code, and explain your reasoning in the final report."
    )

    messages = [
        {"role": "system", "content": system_instruction},
        {"role": "user", "content": f"Inspect '{target_file}', fix any errors, save updates to disk, and verify execution."},
    ]

    print(f"=== Starting Agentic Code Repair Agent ===")
    print(f"Target File: ./workspace/{target_file}\n")

    for turn in range(1, max_turns + 1):
        print(f"--- [Turn {turn}/{max_turns}] Prompting qwen2.5-coder:7b ---")

        response = await OLLAMA_CLIENT.chat.completions.create(
            model="qwen2.5-coder:7b",
            messages=messages,
            tools=TOOLS_SCHEMA,
            temperature=0.1,
        )

        choice = response.choices[0].message
        messages.append(choice.model_dump())

        # Path 1: Native Function Tool Calls
        if choice.tool_calls:
            for tool_call in choice.tool_calls:
                fn_name = tool_call.function.name
                fn_args = json.loads(tool_call.function.arguments)
                print(f"-> Executing Tool: {fn_name}({fn_args.get('file_path', '')})")

                result = TOOL_REGISTRY[fn_name](**fn_args) if fn_name in TOOL_REGISTRY else "Tool not found."
                print(f"   Result: {str(result)[:120]}...\n")

                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": str(result),
                })
            continue

        # Path 2: Raw JSON Fallback Parser
        fallback_fn, fallback_args = extract_fallback_tool_call(choice.content)
        if fallback_fn and fallback_fn in TOOL_REGISTRY:
            print(f"-> Executing Fallback Tool: {fallback_fn}({fallback_args.get('file_path', '')})")
            result = TOOL_REGISTRY[fallback_fn](**fallback_args)
            print(f"   Result: {str(result)[:120]}...\n")

            messages.append({
                "role": "user",
                "content": f"[System Tool Output for {fallback_fn}]:\n{result}",
            })
            continue

        # Path 3: Code Block Fallback Interceptor
        extracted_code = extract_python_code_block(choice.content)
        if extracted_code:
            print(f"-> Intercepted Python code block in text output. Automatically applying to '{target_file}'...")
            write_result = write_workspace_file(target_file, extracted_code)
            print(f"   {write_result}\n")

            # Force auto-verification step into context
            messages.append({
                "role": "user",
                "content": (
                    f"[System Event]: Applied your suggested code to '{target_file}'. "
                    f"Now running execution verification..."
                ),
            })
            
            # Immediately trigger verification on disk
            exec_result = run_python_script(target_file)
            print(f"-> Verification Output:\n{exec_result}\n")
            
            messages.append({
                "role": "user",
                "content": f"[Verification System Result]:\n{exec_result}",
            })
            continue

        # Path 4: Completion Reached
        print("=== Final Diagnostic Report ===")
        print(choice.content)
        break

# Execute on your existing file inside ./workspace/
await inspect_and_fix_code(target_file="orders.py")

=== Starting Agentic Code Repair Agent ===
Target File: ./workspace/orders.py

--- [Turn 1/10] Prompting qwen2.5-coder:7b ---
-> Executing Fallback Tool: read_workspace_file(orders.py)
   Result: from pydantic import BaseModel


class Order(BaseModel):
    customer_id: str
    item: str
    total_price: float
    s...

--- [Turn 2/10] Prompting qwen2.5-coder:7b ---
-> Executing Fallback Tool: write_workspace_file(orders.py)
   Result: Successfully updated and saved 'orders.py'....

--- [Turn 3/10] Prompting qwen2.5-coder:7b ---
-> Executing Fallback Tool: run_python_script(orders.py)
   Result: [EXECUTION SUCCESSFUL]
Script executed cleanly....

--- [Turn 4/10] Prompting qwen2.5-coder:7b ---
=== Final Diagnostic Report ===
All errors and logic bugs have been fixed, and the code has been verified to execute successfully. No further changes are needed.

Final Report:
1. The initial code had a typo in the type hint for the `orders` parameter in the `group_orders_by_customer_id` function, 